In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder

def TimeSeries(days):

    #WE ARE GETTING RID OF REPETITIVE SAME DATES:

    # Example time series data with numerical and categorical columns
    data = {
        'ID': ['ID_5da86e71bf5dee4cf5047046','ID_5da86e71bf5dee4cf5047046','ID_5da86e71bf5dee4cf5047046',
                'ID_5da86e71bf5dee4cf5047046','ID_5da86e71bf5dee4cf5047046','ID_5da86e71bf5dee4cf5047046',
                'ID_5da86e71bf5dee4cf5047046','ID_5da86e71bf5dee4cf5047046','ID_5da86e71bf5dee4cf5047046',
                'ID_5da86e71bf5dee4cf5047046','ID_5ee74f25f865a8154966b412','ID_5ee74f25f865a8154966b412',
                'ID_5ee74f25f865a8154966b412','ID_5ee74f25f865a8154966b412','ID_5ee74f25f865a8154966b412',
                'ID_5ee74f25f865a8154966b412','ID_5ee74f25f865a8154966b412','ID_5ee74f25f865a8154966b412',
                'ID_5ee74f25f865a8154966b412','ID_5ee74f25f865a8154966b412'],

        'date': ['2022-01-01','2022-01-01','2022-01-02','2022-01-03','2022-01-04',
                '2022-01-05','2022-01-05','2022-01-06','2022-01-07','2022-01-08',
                '2022-01-15','2022-01-15','2022-01-16','2022-01-17','2022-01-18',
                '2022-01-22','2022-01-23','2022-01-24','2022-01-26','2022-01-28'],

        'value': [10, 20, 51, 25, 70,11, 21, 16, 26, 31,11, 21, 106, 26, 10,11, 271,58, 27, 33],
        'category': ['B', 'B', 'A', 'C', 'C','B', 'B', 'A', 'C', 'C','D', 'D', 'E', 'C', 'C','B', 'B', 'A', 'D', 'B']
    }


    # Convert data to DataFrame
    df = pd.DataFrame(data)
    
    # Encoding different columns (With label encoder)
    label_enc=['category']
    label_encoder = LabelEncoder()
    
    for column in label_enc:
        encoded_value= label_encoder.fit_transform(df[column])
        df[column]= encoded_value


    unique_ids = df['ID'].unique()

    column_names=df.columns
    
    id_dictionary={}

    count=1
    
    result_df= pd.DataFrame(columns=column_names)
    sorted_df= []
    
    days_count=1
    
    
    for day in range(1, days+1):
        
        for id in unique_ids:
   
            for col in column_names:
                id_dictionary[col]= list(df[df['ID']==id][col])  #inserting column observations belonging to a specific ID

            id_df=pd.DataFrame(id_dictionary)


            # Convert 'date' column to datetime and set as index
            id_df['date'] = pd.to_datetime(id_df['date'])

            #setting date as index
            id_df.set_index('date', inplace=True)

            # Ensure unique index values by grouping and aggregating
            id_df = id_df.groupby(level=0).first()

            def custom_resample(column):
                if pd.api.types.is_numeric_dtype(column):
                    return column.resample('D').mean()  # Resample numerical columns by taking the mean
                elif pd.api.types.is_categorical_dtype(column):
                    return column.resample('D').mode()  # Resample categorical columns by mode filling
                else:
                    return column  # Return unchanged for other column types

            # Apply custom resampling function to each column
            resampled_raw_data = id_df.apply(custom_resample)

        

            resampled_data= resampled_raw_data.copy()
            
            ##############################################################################################
            print("#"*74)

            #CREATING LAG FEATURE COLUMNS
            lag_features_count=4
            lag_feature_names=[]
            all_features=[]
            
            for name in column_names:
                all_features.append(name)
                
            for i in range(1,lag_features_count+1):
                num=str(i)
                resampled_data[f'lag_{num}']=id_df['value'].shift(i)
                lag_feature_names.append(str(f'lag_{num}'))
                all_features.append(str(f'lag_{num}'))

            
            ##############################################################################################
            print("#"*74)

            columns_with_nan= lag_feature_names
            resampled_data= resampled_data.dropna(subset=columns_with_nan)


            ##############################################################################################
            print("#"*74)

            values=list(resampled_data['value'])



            ##############################################################################################
            print("#"*74)

            #predict lag Features
            predict_lag_cols= lag_feature_names.copy()
            
            
            max_date=resampled_data.index.max()
            

            next_date= max_date + pd.Timedelta(days=1)


            predict_features={'date': [next_date], 
                              'ID':[id],
                              'category':[resampled_data[resampled_data.index >= resampled_data.index.max()-pd.DateOffset(days=3)]['category'].mode().iloc[0]]}

            count=count+1



            for i, colu in enumerate(predict_lag_cols):

                predict_features[colu]= [values[-(i+1)]]



            pred_df=pd.DataFrame(predict_features)
            
            

             # Convert 'date' column to datetime and set as index
            pred_df['date'] = pd.to_datetime(pred_df['date'])

            #setting date as index
            pred_df.set_index('date', inplace=True)
           
            
            
            
            ##################################################################################################


            X=resampled_data.drop(columns=['value','ID'],axis=1)
            y=resampled_data['value']
            

            X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.1, random_state=42)

            model=DecisionTreeRegressor(random_state=42)
            model.fit(X_train,y_train)

            y_pred=model.predict(X_test)

            mse_score=mean_squared_error(y_test,y_pred)

            print(mse_score)

            X_pred =pred_df.drop(columns=['ID'],axis=1)
            display(X_pred)
            new_y_pred=model.predict(X_pred)
           

            predict_features=pd.DataFrame({'date': [next_date], 
                              'ID':[id],
                              'value': new_y_pred,
                              'category':[resampled_data[resampled_data.index >= resampled_data.index.max()-pd.DateOffset(13)]['category'].mode().iloc[0]]})

            predict_features=pd.DataFrame(predict_features)

            df=pd.concat([df,predict_features],axis=0)
            
            
            #All features
            all_predict_features=pd.DataFrame({'date': [next_date], 
            'ID':[id],
            'value': new_y_pred,
            'category':[resampled_data[resampled_data.index >= resampled_data.index.max()-pd.DateOffset(13)]['category'].mode().iloc[0]]})

            for i, colu in enumerate(predict_lag_cols):

                all_predict_features[colu]= [values[-(i+1)]]


            if days_count==7 or days_count==14:
                result_df=result_df.append(all_predict_features,ignore_index=True)
                
        days_count= days_count+1
            
    return result_df


outcome=TimeSeries(14)
dic=[]
#display(outcome)

records_dic=outcome.to_dict(orient='records')
list_dic=outcome.to_dict(orient='list')



print('\n')

unique_ids=outcome.columns

for id in unique_ids:
    for row in records_dic:
    
        if row['ID']==id:
            dic.append(row)
            
print('\n')
final=pd.DataFrame(dic)

display(final)


##########################################################################
##########################################################################
##########################################################################
##########################################################################
100.0


,category,lag_1,lag_2,lag_3,lag_4
date,,,,,
2022-01-09,2,31,26,16,11


##########################################################################
##########################################################################
##########################################################################
##########################################################################
67600.0


,category,lag_1,lag_2,lag_3,lag_4
date,,,,,
2022-01-29,1.0,33.0,27.0,58.0,271.0


##########################################################################
##########################################################################
##########################################################################
##########################################################################
100.0


...

##########################################################################
##########################################################################
##########################################################################
##########################################################################
225.0


,category,lag_1,lag_2,lag_3,lag_4
date,,,,,
2022-01-22,2.0,31.0,31.0,31.0,31.0


##########################################################################
##########################################################################
##########################################################################
##########################################################################
28564.0


,category,lag_1,lag_2,lag_3,lag_4
date,,,,,
2022-02-11,1.0,33.0,33.0,33.0,33.0


""


In [10]:
IDs=[]
clicks=[]


for row in final.to_dict(orient='records'):
    ID=str(row['ID'])+'_'+str(row['date'])[:-9].replace('-','_')
    click=row['value']
    IDs.append(ID)
    clicks.append(click)

sorted_predictions=pd.DataFrame({'ID':IDs, 'clicks':clicks})
display(sorted_predictions)

,ID,clicks
